In [ ]:
"""
Multi-CNN Ensemble & Feature Fusion for Leaf Classification
Builds on the single-model EfficientNetB0 pipeline.

Two combination strategies:
A) Prediction-level ensemble (soft voting / weighted averaging)
B) Feature-level fusion (concatenate embeddings -> joint classifier)
"""

import os
import kagglehub
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# ----------------------------------------------------------------------
# 1. Dataset (same as before)
# ----------------------------------------------------------------------
path = kagglehub.dataset_download("csafrit2/plant-leaves-for-image-classification")

def find_split_dirs(root):
    found = {}
    for dirpath, dirnames, _ in os.walk(root):
        for d in dirnames:
            key = d.lower()
            if key in ("train", "valid", "validation", "test"):
                found[key] = os.path.join(dirpath, d)
    return found

split_dirs = find_split_dirs(path)
train_dir = split_dirs.get("train")
val_dir = split_dirs.get("valid") or split_dirs.get("validation")
test_dir = split_dirs.get("test")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    train_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED, shuffle=True
)
val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    val_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED, shuffle=False
) if val_dir else None
test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    test_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED, shuffle=False
) if test_dir else None

class_names = train_ds_raw.class_names
num_classes = len(class_names)
print(f"Found {num_classes} classes:", class_names)

if val_ds_raw is None:
    val_batches = int(0.2 * tf.data.experimental.cardinality(train_ds_raw).numpy())
    val_ds_raw = train_ds_raw.take(val_batches)
    train_ds_raw = train_ds_raw.skip(val_batches)

AUTOTUNE = tf.data.AUTOTUNE

# IMPORTANT: each backbone was pretrained with a different preprocessing
# convention (EfficientNet: raw 0-255, ResNet: caffe-style mean subtraction,
# MobileNetV2: scaled to [-1, 1]). We keep one shared "raw" dataset and
# apply each model's correct preprocess_input inside that model's own
# input pipeline, so all three stay correctly normalized.
def prep(ds, preprocess_fn, training=False):
    if preprocess_fn is not None:
        ds = ds.map(lambda x, y: (preprocess_fn(x), y), num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.cache().shuffle(1000).prefetch(AUTOTUNE)
    else:
        ds = ds.cache().prefetch(AUTOTUNE)
    return ds

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
], name="data_augmentation")

# ----------------------------------------------------------------------
# 2. Model factory — build any backbone with a matching head
# ----------------------------------------------------------------------
BACKBONES = {
    "efficientnetb0": {
        "fn": tf.keras.applications.EfficientNetB0,
        "preprocess": None,  # normalization is baked into the model
    },
    "resnet50": {
        "fn": tf.keras.applications.ResNet50,
        "preprocess": tf.keras.applications.resnet50.preprocess_input,
    },
    "mobilenetv2": {
        "fn": tf.keras.applications.MobileNetV2,
        "preprocess": tf.keras.applications.mobilenet_v2.preprocess_input,
    },
}

def build_model(name, num_classes, fine_tune_layers=30):
    cfg = BACKBONES[name]
    base = cfg["fn"](input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
    base.trainable = False

    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = data_augmentation(inputs)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D(name="gap_features")(x)  # named -> used for feature fusion later
    feat = layers.Dropout(0.3)(x)
    y = layers.Dense(128, activation="relu")(feat)
    y = layers.Dropout(0.2)(y)
    outputs = layers.Dense(num_classes, activation="softmax")(y)

    model = tf.keras.Model(inputs, outputs, name=f"leaf_{name}")
    return model, base

def compile_head(model, lr=1e-3):
    model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])

def compile_finetune(model, lr=1e-5):
    model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])

def make_callbacks(ckpt_path):
    return [
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor="val_loss"),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7),
        tf.keras.callbacks.ModelCheckpoint(ckpt_path, save_best_only=True, monitor="val_loss"),
    ]

# ----------------------------------------------------------------------
# 3. Train each backbone (two-phase: head -> fine-tune)
# ----------------------------------------------------------------------
EPOCHS_HEAD = 15
EPOCHS_FINE_TUNE = 15

trained_models = {}
histories = {}

for name in BACKBONES:
    print(f"\n{'='*60}\nTraining {name}\n{'='*60}")

    preprocess_fn = BACKBONES[name]["preprocess"]
    train_ds = prep(train_ds_raw, preprocess_fn, training=True)
    val_ds = prep(val_ds_raw, preprocess_fn, training=False)

    model, base = build_model(name, num_classes)
    compile_head(model)
    cbs = make_callbacks(f"best_{name}.keras")

    hist1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, callbacks=cbs)

    base.trainable = True
    fine_tune_at = len(base.layers) - 30
    for layer in base.layers[:fine_tune_at]:
        layer.trainable = False
    compile_finetune(model)

    hist2 = model.fit(
        train_ds, validation_data=val_ds,
        epochs=EPOCHS_HEAD + EPOCHS_FINE_TUNE,
        initial_epoch=hist1.epoch[-1] + 1,
        callbacks=cbs,
    )

    trained_models[name] = model
    histories[name] = (hist1, hist2)
    model.save(f"leaf_{name}_final.keras")
    print(f"Saved leaf_{name}_final.keras")

# ----------------------------------------------------------------------
# 4A. Prediction-level ensemble (soft voting)
# ----------------------------------------------------------------------
def get_test_ds_for(name):
    preprocess_fn = BACKBONES[name]["preprocess"]
    return prep(test_ds_raw, preprocess_fn, training=False)

def collect_predictions(model, ds):
    """Returns (probs, y_true) aligned in the same order."""
    all_probs, all_labels = [], []
    for images, labels in ds:
        probs = model.predict(images, verbose=0)
        all_probs.append(probs)
        all_labels.append(labels.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)

per_model_probs = {}
y_true_ref = None
for name, model in trained_models.items():
    ds = get_test_ds_for(name)
    probs, y_true = collect_predictions(model, ds)
    per_model_probs[name] = probs
    if y_true_ref is None:
        y_true_ref = y_true
    else:
        assert np.array_equal(y_true, y_true_ref), "Label order mismatch across datasets — shuffle must stay False"

# Simple average (equal-weight soft voting)
avg_probs = np.mean(list(per_model_probs.values()), axis=0)
ensemble_pred = np.argmax(avg_probs, axis=1)

print("\n--- Prediction-level Ensemble (soft voting, equal weights) ---")
print(classification_report(y_true_ref, ensemble_pred, target_names=class_names))

# Optional: weighted average using each model's own val accuracy as weight
val_accs = {}
for name, model in trained_models.items():
    ds = prep(val_ds_raw, BACKBONES[name]["preprocess"], training=False)
    _, acc = model.evaluate(ds, verbose=0)
    val_accs[name] = acc

weights = np.array([val_accs[n] for n in per_model_probs])
weights = weights / weights.sum()
weighted_probs = sum(w * per_model_probs[n] for w, n in zip(weights, per_model_probs))
weighted_pred = np.argmax(weighted_probs, axis=1)

print("\n--- Prediction-level Ensemble (weighted by val accuracy) ---")
print("Weights:", dict(zip(per_model_probs.keys(), weights)))
print(classification_report(y_true_ref, weighted_pred, target_names=class_names))

cm = confusion_matrix(y_true_ref, weighted_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=False, cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix — Weighted Ensemble")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.tight_layout()
plt.savefig("confusion_matrix_ensemble.png", dpi=150)
plt.show()

# ----------------------------------------------------------------------
# 4B. Feature-level fusion (concatenate embeddings, train joint head)
# ----------------------------------------------------------------------
# Build a feature-extractor for each backbone (output = GAP layer, pre-head)
feature_extractors = {
    name: tf.keras.Model(model.input, model.get_layer("gap_features").output)
    for name, model in trained_models.items()
}

def extract_features(ds_raw, split_name):
    """
    Runs each backbone's feature extractor over the SAME raw images
    (each with its own correct preprocessing), then concatenates
    the resulting embeddings per image.
    """
    feats_per_model = {}
    labels_ref = None
    for name, extractor in feature_extractors.items():
        ds = prep(ds_raw, BACKBONES[name]["preprocess"], training=False)
        feats, labels = [], []
        for images, y in ds:
            f = extractor.predict(images, verbose=0)
            feats.append(f)
            labels.append(y.numpy())
        feats = np.concatenate(feats)
        labels = np.concatenate(labels)
        feats_per_model[name] = feats
        if labels_ref is None:
            labels_ref = labels
        else:
            assert np.array_equal(labels, labels_ref), f"Label order mismatch in {split_name}"
    fused = np.concatenate(list(feats_per_model.values()), axis=1)
    print(f"{split_name} fused feature shape: {fused.shape}")
    return fused, labels_ref

X_train_fused, y_train_fused = extract_features(train_ds_raw, "train")
X_val_fused, y_val_fused = extract_features(val_ds_raw, "val")
X_test_fused, y_test_fused = extract_features(test_ds_raw, "test")

# Small dense classifier on top of the fused feature vector
fusion_head = models.Sequential([
    layers.Input(shape=(X_train_fused.shape[1],)),
    layers.Dropout(0.4),
    layers.Dense(256, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation="softmax"),
], name="fusion_head")

fusion_head.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                     loss="sparse_categorical_crossentropy", metrics=["accuracy"])

fusion_cbs = [
    tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True, monitor="val_loss"),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-7),
]

fusion_history = fusion_head.fit(
    X_train_fused, y_train_fused,
    validation_data=(X_val_fused, y_val_fused),
    epochs=50,
    batch_size=BATCH_SIZE,
    callbacks=fusion_cbs,
)

fusion_test_loss, fusion_test_acc = fusion_head.evaluate(X_test_fused, y_test_fused, verbose=0)
print(f"\nFeature Fusion — Test accuracy: {fusion_test_acc:.4f} | Test loss: {fusion_test_loss:.4f}")

fusion_pred = np.argmax(fusion_head.predict(X_test_fused, verbose=0), axis=1)
print("\n--- Feature-level Fusion classification report ---")
print(classification_report(y_test_fused, fusion_pred, target_names=class_names))

cm_fusion = confusion_matrix(y_test_fused, fusion_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_fusion, annot=False, cmap="Greens", xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix — Feature Fusion")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.tight_layout()
plt.savefig("confusion_matrix_fusion.png", dpi=150)
plt.show()

fusion_head.save("leaf_fusion_head.keras")

# ----------------------------------------------------------------------
# 5. Final comparison summary
# ----------------------------------------------------------------------
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
for name in trained_models:
    ds = get_test_ds_for(name)
    _, acc = trained_models[name].evaluate(ds, verbose=0)
    print(f"{name:15s} standalone test acc: {acc:.4f}")

ensemble_acc = np.mean(ensemble_pred == y_true_ref)
weighted_acc = np.mean(weighted_pred == y_true_ref)
print(f"{'Soft-voting':15s} ensemble test acc: {ensemble_acc:.4f}")
print(f"{'Weighted vote':15s} ensemble test acc: {weighted_acc:.4f}")
print(f"{'Feature fusion':15s} test acc: {fusion_test_acc:.4f}")

Using Colab cache for faster access to the 'plant-leaves-for-image-classification' dataset.
Found 4274 files belonging to 22 classes.
Found 110 files belonging to 22 classes.
Found 110 files belonging to 22 classes.
Found 22 classes: ['Alstonia Scholaris diseased (P2a)', 'Alstonia Scholaris healthy (P2b)', 'Arjun diseased (P1a)', 'Arjun healthy (P1b)', 'Bael diseased (P4b)', 'Basil healthy (P8)', 'Chinar diseased (P11b)', 'Chinar healthy (P11a)', 'Gauva diseased (P3b)', 'Gauva healthy (P3a)', 'Jamun diseased (P5b)', 'Jamun healthy (P5a)', 'Jatropha diseased (P6b)', 'Jatropha healthy (P6a)', 'Lemon diseased (P10b)', 'Lemon healthy (P10a)', 'Mango diseased (P0b)', 'Mango healthy (P0a)', 'Pomegranate diseased (P9b)', 'Pomegranate healthy (P9a)', 'Pongamia Pinnata diseased (P7b)', 'Pongamia Pinnata healthy (P7a)']

Training efficientnetb0
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Epoch 1/15
129/134 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.4184 - loss: 2.0197